# bansal
# CrimeSync Intelligence Engine: Graph Neural Network Link Prediction & Mule Account Detection
### Advanced GNN Pipeline (GraphSAGE / RGCN) for Crime Syndicate Community Detection and Link Inference

In [ ]:
# bansal
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import IsolationForest

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
print("[+] PyTorch & Graph Neural Network backend initialized on:", "cuda" if torch.cuda.is_available() else "cpu")

### 1. Heterogeneous Graph Construction & Feature Engineering
Constructing graph representation with nodes: `Suspect`, `BankAccount`, `DeviceHash`, `Location` and edges: `TRANSFERRED_TO`, `USED_DEVICE`, `CALL_LOG`.

In [ ]:
# bansal
class CrimeGraphDataset:
    def __init__(self, num_suspects=500, num_accounts=1200, embedding_dim=128):
        self.num_suspects = num_suspects
        self.num_accounts = num_accounts
        self.embedding_dim = embedding_dim
        
        # Synthetic Node Embeddings (Financial velocity, risk scores, call frequency)
        self.suspect_features = torch.randn(num_suspects, embedding_dim)
        self.account_features = torch.randn(num_accounts, embedding_dim)
        
        # Adjacency matrices
        self.edge_index_transfer = torch.randint(0, num_accounts, (2, 3500))
        self.edge_index_suspect_account = torch.stack([
            torch.randint(0, num_suspects, (2000,)),
            torch.randint(0, num_accounts, (2000,))
        ])
        
    def get_data(self):
        return {
            'x_suspect': self.suspect_features,
            'x_account': self.account_features,
            'edge_transfer': self.edge_index_transfer,
            'edge_ownership': self.edge_index_suspect_account
        }

dataset = CrimeGraphDataset()
graph_data = dataset.get_data()
print(f"[+] Graph Loaded: {dataset.num_suspects} suspects, {dataset.num_accounts} accounts, {graph_data['edge_transfer'].shape[1]} transaction edges")

### 2. Multi-Relational Graph Neural Network (CrimeSync-GNN Architecture)

In [ ]:
# bansal
class GraphSAGEConv(nn.Module):
    """Neighborhood Aggregator Convolution Layer for Criminal Network Analysis"""
    def __init__(self, in_features, out_features, aggregator='mean'):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.aggregator = aggregator
        self.weight = nn.Linear(in_features * 2, out_features, bias=True)
        self.dropout = nn.Dropout(0.25)
        
    def forward(self, x, edge_index):
        src, dst = edge_index[0], edge_index[1]
        num_nodes = x.size(0)
        
        # Scatter mean message aggregation
        aggr_msg = torch.zeros(num_nodes, self.in_features, device=x.device)
        aggr_msg.index_add_(0, dst, x[src])
        
        degree = torch.bincount(dst, minlength=num_nodes).clamp(min=1).unsqueeze(1)
        aggr_msg = aggr_msg / degree
        
        combined = torch.cat([x, aggr_msg], dim=1)
        out = F.leaky_relu(self.weight(combined), negative_slope=0.2)
        return self.dropout(out)

class CrimeLinkPredictor(nn.Module):
    """Predicts hidden links between kingpins, mule accounts, and fraud operations"""
    def __init__(self, in_channels=128, hidden_channels=64, out_channels=32):
        super().__init__()
        self.conv1 = GraphSAGEConv(in_channels, hidden_channels)
        self.conv2 = GraphSAGEConv(hidden_channels, out_channels)
        self.link_classifier = nn.Sequential(
            nn.Linear(out_channels * 2, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
    def encode(self, x, edge_index):
        h = self.conv1(x, edge_index)
        z = self.conv2(h, edge_index)
        return z
        
    def decode(self, z, edge_pairs):
        src, dst = edge_pairs[0], edge_pairs[1]
        edge_repr = torch.cat([z[src], z[dst]], dim=-1)
        return self.link_classifier(edge_repr).squeeze(-1)

model = CrimeLinkPredictor(in_channels=128, hidden_channels=64, out_channels=32)
print(model)

### 3. Training Loop with Contrastive Loss & Negative Sampling

In [ ]:
# bansal
optimizer = torch.optim.AdamW(model.parameters(), lr=0.005, weight_decay=1e-4)
criterion = nn.BCELoss()

# Simulated Training Epochs
for epoch in range(1, 11):
    model.train()
    optimizer.zero_grad()
    
    z = model.encode(graph_data['x_account'], graph_data['edge_transfer'])
    
    # Positive edges
    pos_edge = graph_data['edge_transfer'][:, :500]
    pos_pred = model.decode(z, pos_edge)
    pos_target = torch.ones_like(pos_pred)
    
    # Negative sampling edges
    neg_src = torch.randint(0, dataset.num_accounts, (500,))
    neg_dst = torch.randint(0, dataset.num_accounts, (500,))
    neg_edge = torch.stack([neg_src, neg_dst])
    neg_pred = model.decode(z, neg_edge)
    neg_target = torch.zeros_like(neg_pred)
    
    loss = criterion(pos_pred, pos_target) + criterion(neg_pred, neg_target)
    loss.backward()
    optimizer.step()
    
    auc = roc_auc_score(torch.cat([pos_target, neg_target]).detach().numpy(), 
                        torch.cat([pos_pred, neg_pred]).detach().numpy())
    if epoch % 2 == 0:
        print(f"Epoch {epoch:02d} | Loss: {loss.item():.4f} | Validation ROC-AUC: {auc:.4f}")

### 4. Mule Account Isolation Forest & Flow Velocity Scoring

In [ ]:
# bansal
def detect_mule_layering(account_embeddings, contamination=0.03):
    iso_forest = IsolationForest(n_estimators=200, contamination=contamination, random_state=42)
    anomaly_labels = iso_forest.fit_predict(account_embeddings.detach().numpy())
    anomaly_scores = -iso_forest.decision_function(account_embeddings.detach().numpy())
    
    flagged_indices = np.where(anomaly_labels == -1)[0]
    print(f"[!] Flagged {len(flagged_indices)} High-Risk Layering / Mule Accounts for Investigation Dossier.")
    return flagged_indices, anomaly_scores

mules, scores = detect_mule_layering(z)
print("Top 5 suspicious node IDs:", mules[:5])